## **Extensive Cleaning Method**

In [1]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

In [2]:
spark = sparknlp.start()
print("Spark NLP version", sparknlp.version())
print("Apache Spark version:", spark.version)

26/04/14 14:55:00 WARN Utils: Your hostname, MacBook-Pro-4.local resolves to a loopback address: 127.0.0.1; using 10.18.20.202 instead (on interface en0)
26/04/14 14:55:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/leena/Desktop/tweet-sentiment-analysis/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/leena/.ivy2/cache
The jars for the packages stored in: /Users/leena/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-547c6ca5-bede-4224-b5e1-ad13e3670b07;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in centra

26/04/14 14:55:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark NLP version 6.3.3
Apache Spark version: 3.3.1


In [3]:
train_df = spark.read.csv('../data/twitter_training.csv')
train_df.show(5)

+----+-----------+--------+--------------------+
| _c0|        _c1|     _c2|                 _c3|
+----+-----------+--------+--------------------+
|2401|Borderlands|Positive|im getting on bor...|
|2401|Borderlands|Positive|I am coming to th...|
|2401|Borderlands|Positive|im getting on bor...|
|2401|Borderlands|Positive|im coming on bord...|
|2401|Borderlands|Positive|im getting on bor...|
+----+-----------+--------+--------------------+
only showing top 5 rows



In [4]:
# Add column names
col_names = ["id", "topic", "sentiment", "content"]
train_df = train_df.toDF(*col_names)
train_df.show(5)

+----+-----------+---------+--------------------+
|  id|      topic|sentiment|             content|
+----+-----------+---------+--------------------+
|2401|Borderlands| Positive|im getting on bor...|
|2401|Borderlands| Positive|I am coming to th...|
|2401|Borderlands| Positive|im getting on bor...|
|2401|Borderlands| Positive|im coming on bord...|
|2401|Borderlands| Positive|im getting on bor...|
+----+-----------+---------+--------------------+
only showing top 5 rows



In [5]:
# Convert to document type
documentAssembler = DocumentAssembler() \
    .setInputCol("content") \
    .setOutputCol("document")

In [6]:
# Tokenize the text
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("tokens")

In [7]:
# Correct spellings
spellChecker = NorvigSweetingModel.pretrained() \
    .setInputCols(["tokens"]) \
    .setOutputCol("spell_checked")

spellcheck_norvig download started this may take some time.
Approximate size to download 4.2 MB
[ | ]26/04/14 14:55:25 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/04/14 14:55:25 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
spellcheck_norvig download started this may take some time.
Approximate size to download 4.2 MB
Download done! Loading the resource.


[ / ]

[OK!]


In [8]:
# Normalize
normalizer = Normalizer() \
    .setInputCols(["spell_checked"]) \
    .setOutputCol("normalized")

In [9]:
# Remove stopwords
stopwordsCleaner = StopWordsCleaner() \
    .setInputCols(["normalized"]) \
    .setOutputCol("cleaned")

In [10]:
# Reduce to lemmas
lemmatizer = LemmatizerModel.pretrained() \
    .setInputCols(["cleaned"]) \
    .setOutputCol("lemmas")

lemma_antbnc download started this may take some time.
26/04/14 14:55:30 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
Approximate size to download 907.6 KB
[ | ]26/04/14 14:55:30 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/04/14 14:55:31 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
lemma_antbnc download started this may take some time.
Approximate size

In [11]:
# To get the final cleaned content as text
finisher = Finisher() \
    .setInputCols(["lemmas"]) \
    .setOutputCols(["cleaned_text"]) \
    .setOutputAsArray(False) \
    .setCleanAnnotations(False)\
    .setAnnotationSplitSymbol(" ")  

In [12]:
# Setup the pipeline
nlp_pipeline = Pipeline().setStages([
    documentAssembler, tokenizer, spellChecker, normalizer, stopwordsCleaner, lemmatizer, finisher
])

In [13]:
# Run data through pipeline
model = nlp_pipeline.fit(train_df)
clean_train_df = model.transform(train_df)
clean_train_df.show()

+----+-----------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|  id|      topic|sentiment|             content|            document|              tokens|       spell_checked|          normalized|             cleaned|              lemmas|        cleaned_text|
+----+-----------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|2401|Borderlands| Positive|im getting on bor...|[{document, 0, 52...|[{token, 0, 1, im...|[{token, 0, 1, im...|[{token, 0, 1, im...|[{token, 0, 1, im...|[{token, 0, 1, im...|im get borderland...|
|2401|Borderlands| Positive|I am coming to th...|[{document, 0, 50...|[{token, 0, 0, I,...|[{token, 0, 0, I,...|[{token, 0, 0, I,...|[{token, 5, 10, c...|[{token, 5, 10, c...|    come border kill|
|2401|Borderlan

In [14]:
clean_train_df.select("cleaned_text").show()

+--------------------+
|        cleaned_text|
+--------------------+
|im get borderland...|
|    come border kill|
|im get borderland...|
|im come borderlan...|
|im get borderland...|
|im get borderland...|
|spend hour make s...|
|spend couple hour...|
|spend hour someth...|
|spend hour make s...|
|spend hour make s...|
|                    |
|RockHard La Varlo...|
|RockHard La Varlo...|
|RockHard La Varlo...|
|RockHard La Vita ...|
|Live Rock Hard mu...|
|IHard like RARE L...|
|first borderland ...|
|first Borderlands...|
+--------------------+
only showing top 20 rows



### **Export cleaned data to CSV**

In [15]:
export_df = clean_train_df.select(col("id"), 
                                  col("topic"), 
                                  col("sentiment").alias("og_sentiment"), 
                                  col("cleaned_text"))
export_df.show()

+----+-----------+------------+--------------------+
|  id|      topic|og_sentiment|        cleaned_text|
+----+-----------+------------+--------------------+
|2401|Borderlands|    Positive|im get borderland...|
|2401|Borderlands|    Positive|    come border kill|
|2401|Borderlands|    Positive|im get borderland...|
|2401|Borderlands|    Positive|im come borderlan...|
|2401|Borderlands|    Positive|im get borderland...|
|2401|Borderlands|    Positive|im get borderland...|
|2402|Borderlands|    Positive|spend hour make s...|
|2402|Borderlands|    Positive|spend couple hour...|
|2402|Borderlands|    Positive|spend hour someth...|
|2402|Borderlands|    Positive|spend hour make s...|
|2402|Borderlands|    Positive|spend hour make s...|
|2402|Borderlands|    Positive|                    |
|2403|Borderlands|     Neutral|RockHard La Varlo...|
|2403|Borderlands|     Neutral|RockHard La Varlo...|
|2403|Borderlands|     Neutral|RockHard La Varlo...|
|2403|Borderlands|     Neutral|RockHard La Vit

In [16]:
export_df.toPandas().to_csv("../cleaned_data/nlp_data.csv", index=False)